In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [ ]:
BASE_DIR   = os.path.dirname(os.path.abspath(__file__))
BACKEND    = os.path.join(BASE_DIR, "..", "backend")
CSV_PATH   = os.path.join(BACKEND, "solar_data.csv")
MODEL_PATH = os.path.join(BACKEND, "model.pkl")

In [ ]:
if not os.path.exists(CSV_PATH):
    print(f"\n[ERR] solar_data.csv not found at: {CSV_PATH}")
    print("[ERR] Export it first from: http://localhost:5000/api/export")
    exit(1)

df = pd.read_csv(CSV_PATH)
print(f"\n[INFO] Loaded {len(df)} rows from {CSV_PATH}")
print(f"[INFO] Columns: {list(df.columns)}")
print(f"\n[INFO] Sample data:")
print(df.head())

In [ ]:
original_len = len(df)

df = df.dropna()
df = df[df["voltage"]     >= 0]
df = df[df["current"]     >= 0]
df = df[df["temperature"] > -40]
df = df[df["temperature"] < 100]
df = df[df["light"]       >= 0]
df = df[df["power"]       >= 0]

print(f"\n[INFO] Rows after cleaning: {len(df)} (removed {original_len - len(df)} bad rows)")

In [ ]:
RATED_MAX_POWER = 6

df["efficiency"] = (df["power"] / RATED_MAX_POWER) * 100
df["efficiency"] = df["efficiency"].clip(0, 100)

print(f"\n[INFO] Efficiency stats:")
print(f"  Min : {df['efficiency'].min():.2f}%")
print(f"  Max : {df['efficiency'].max():.2f}%")
print(f"  Mean: {df['efficiency'].mean():.2f}%")

In [ ]:
X = df[["voltage", "current", "temperature", "light"]]
y = df["efficiency"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n[INFO] Training samples : {len(X_train)}")
print(f"[INFO] Testing samples  : {len(X_test)}")

In [ ]:
print("\n[INFO] Training Random Forest model...")

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)
print("[OK]  Training complete")


In [ ]:
predictions = model.predict(X_test)

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)

print("\n" + "=" * 50)
print("  Model Accuracy Results")
print("=" * 50)
print(f"  MAE  (avg error)     : {mae:.2f}%")
print(f"  RMSE (error spread)  : {rmse:.2f}%")
print(f"  R²   (accuracy 0-1)  : {r2:.4f}")
print()

if r2 >= 0.95:
    print("  [EXCELLENT] Model is very accurate")
elif r2 >= 0.85:
    print("  [GOOD] Model is working well")
elif r2 >= 0.70:
    print("  [OK] Model is acceptable, collect more data to improve")
else:
    print("  [POOR] Collect more data across varied conditions")

In [ ]:
importance = pd.DataFrame({
    "feature"   : ["voltage", "current", "temperature", "light"],
    "importance": model.feature_importances_ * 100
}).sort_values("importance", ascending=False)

print("\n[INFO] Feature importance (which sensors matter most):")
for _, row in importance.iterrows():
    bar = "█" * int(row["importance"] / 2)
    print(f"  {row['feature']:<12} {row['importance']:5.1f}%  {bar}")

In [ ]:
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print(f"\n[OK]  Model saved → {MODEL_PATH}")
print("[OK]  Restart Flask (backend/app.py) to load the new model")

In [ ]:
print("\n" + "=" * 50)
print("  Test Prediction")
print("=" * 50)

sample = pd.DataFrame([{
    "voltage"    : 15.0,
    "current"    : 1.2,
    "temperature": 35.0,
    "light"      : 75.0
}])

result = model.predict(sample)[0]
print(f"  Input  → V:15.0  I:1.2  T:35°C  L:75%")
print(f"  Output → Predicted efficiency: {result:.2f}%")